# **Elastic Net Regression from Scratch**

In [44]:
from math import log1p

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    root_mean_squared_error,
    mean_squared_error,
    r2_score
)
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing

In [45]:
class ElasticNet:

    def __init__(self , learning_rate , alpha , epochs , l1_ratio):
        self.learning_rate = learning_rate
        self.alpha = alpha
        self.epochs = epochs
        self.l1_ratio = l1_ratio

        self.w = None
        self.b = None
        self.loss_history = []

    def fit(self , X_train , y_train):

        n_samples , n_features = X_train.shape

        self.w = np.zeros(shape = X_train.shape[1])
        self.b = 0

        for epoch in range(self.epochs):

            # Predicted Value
                y_hat = X_train @ self.w + self.b
                error = y_hat - y_train

            # Gradient for " w "
                dw = ((2 / n_samples) * (X_train.T @ error)) + self.alpha * (self.l1_ratio * (np.sign(self.w)) + 2 * (1 - self.l1_ratio) * self.w)

            # Gradient for " b "
                db = (2 / n_samples) * (np.sum(error))

            #L1 penalty
                l1 = self.alpha * self.l1_ratio * np.sum(np.abs(self.w))

            # L2 penalty
                l2 = self.alpha * (1 - self.l1_ratio) * np.sum(self.w ** 2)

            # Loss Calculation
                mse = (1 / n_samples) * (np.sum(error ** 2))
                loss = mse + l1 + l2
                if epoch % 100 == 0:
                    self.loss_history.append(loss)

            # Updating " w " and " b "
                self.w -= self.learning_rate * dw
                self.b -= self.learning_rate * db


    def predict(self , X):
            return X @ self.w + self.b

In [46]:
df_raw = fetch_california_housing(as_frame=True)
df = df_raw.frame

df.head(10)

df = df.rename(
        columns={
            'MedHouseVal' : 'target'
        }
)

X = df.drop('target' , axis = 1)
y = df['target']


X_train , X_test , y_train , y_test = train_test_split(

    X,
    y,
    shuffle=True,
    random_state=42,
    test_size=0.2
)

scaler = StandardScaler(with_mean= True , with_std= True)

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

model = ElasticNet(
    learning_rate = 0.1,
    alpha = 0.01,
    epochs = 100000,
    l1_ratio = 0.5
)

model.fit(X_train , y_train)

y_pred = model.predict(X_test)

print(f"RMSE : {root_mean_squared_error(y_test , y_pred)}")
print(f"MSE : {mean_squared_error(y_test , y_pred)}")
print(f"R2_Score : {r2_score(y_test , y_pred)}")



RMSE : 0.7425266573287123
MSE : 0.5513458368437509
R2_Score : 0.579256670246658
